# 00 — Diagnostik: kemiripan gambar test bermasalah vs data train (embedding CNN)

Notebook **terpisah**, bukan bagian dari alur NB01-04 resmi. Menjawab satu pertanyaan spesifik: 38
gambar test yang prediksinya berubah di ensemble percobaan 1 (mayoritas pola 0/1 -> 2, terutama
kerajinan kertas berbentuk bunga & kemasan bertema makanan) — apakah data TRAIN punya cukup contoh
visual/konseptual serupa berlabel benar, sehingga model **bisa** dilatih untuk menangani pola ini?

**Versi kedua**: percobaan pertama pakai pHash (perceptual hash) untuk mencari tetangga terdekat,
tapi hasilnya tidak valid — jarak hamming tetangga terdekat ada di kisaran 10-18 dari 64 bit
(vs threshold duplikat asli di proyek ini yaitu <=4), artinya itu bukan gambar yang benar-benar
mirip, cuma "paling tidak beda" di antara 26 ribu gambar. pHash didesain mendeteksi duplikat
(foto yang sama, di-crop/resize), BUKAN kemiripan konseptual/material antar foto yang berbeda.

**Perbaikannya**: pakai **embedding CNN** (fitur dari backbone ImageNet-pretrained, BUKAN hasil
fine-tune kompetisi — supaya ruang fiturnya tetap sensitif ke tekstur/material umum, tidak
"dipaksa" cuma memisahkan 3 kelas) + cosine similarity. Ini menangkap kemiripan tekstur/bentuk/
material yang sebenarnya relevan, bukan cuma kemiripan layout piksel kasar.

**Bukti final ada di bagian paling bawah**: grid gambar query + tetangga terdekatnya, supaya bisa
dinilai dengan mata sendiri, bukan cuma angka.


In [ ]:
# timm tidak bawaan Colab -- dipasang dulu di sel terpisah, SEBELUM numpy/pandas/torch lain
# ter-load ulang, supaya tidak memicu error "numpy.dtype size changed" (ketidakcocokan biner).
!pip install -q timm==1.0.11


**PENTING sebelum lanjut**: instalasi di atas kadang membuat numpy/pandas yang sudah ter-load di
proses Colab jadi tidak konsisten secara biner. Sel di bawah SENGAJA mem-restart proses kernel
secara paksa supaya numpy/pandas ter-load ulang bersih dari disk -- ini normal, bukan crash. Setelah
reconnect, lanjutkan dari sel import di bawahnya -- TIDAK perlu menjalankan ulang sel pip install.


In [ ]:
import os
os.kill(os.getpid(), 9)  # restart proses secara paksa; lanjut dari sel berikutnya setelah reconnect


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import timm.data
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)

DRIVE_ROOT = Path("/content/drive/MyDrive/Big Data Challenge - Satria Data 2026")
TRAIN_PROCESSED_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "processed"
FOLD_ASSIGNMENT_PATH = DRIVE_ROOT / "Preprocessing_Output" / "manifests" / "fold_assignment.csv"
TEST_ROOT = DRIVE_ROOT / "BDC2026" / "test"
EVIDENCE_OUTPUT_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "diagnostik_hard_cases"
EVIDENCE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for _p in [TRAIN_PROCESSED_ROOT, FOLD_ASSIGNMENT_PATH, TEST_ROOT]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} tidak ketemu -- cek Drive ter-mount dan path benar.")

label_names = {0: "Recyclable", 1: "Electronic", 2: "Organic"}


## Salin gambar ke disk lokal Colab dulu (hindari bottleneck Drive)

Sama seperti versi pHash sebelumnya -- forward-pass ~26 ribu gambar langsung dari Drive FUSE mount
itu bottleneck I/O yang sama; disalin dulu ke disk lokal Colab secara paralel (resumable, per-file
timeout), baru diproses dari situ.


In [ ]:
LOCAL_STAGING_ROOT = Path("/content/local_staging")
LOCAL_PROCESSED_ROOT = LOCAL_STAGING_ROOT / "processed"
LOCAL_TEST_ROOT = LOCAL_STAGING_ROOT / "test"
LOCAL_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_TEST_ROOT.mkdir(parents=True, exist_ok=True)

N_WORKERS = 64
COPY_TIMEOUT_SECONDS = 60  # per file -- kalau lebih dari ini kemungkinan Drive macet, skip & lanjut


def parallel_copy(pairs, desc="copy", n_workers=N_WORKERS, per_file_timeout=COPY_TIMEOUT_SECONDS, skip_existing=True):
    def _copy_one(src, dest):
        dest = Path(dest)
        if skip_existing and dest.exists() and dest.stat().st_size > 0:
            return
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)

    errors = []
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_copy_one, src, dest): (src, dest) for src, dest in pairs}
        for future in tqdm(list(futures), total=len(futures), desc=f"{desc} ({n_workers} workers)"):
            src, dest = futures[future]
            try:
                future.result(timeout=per_file_timeout)
            except Exception as e:
                errors.append({"src": str(src), "dest": str(dest), "error": str(e)})
    return errors


fold_assignment = pd.read_csv(FOLD_ASSIGNMENT_PATH)
label_by_id = fold_assignment.set_index("image_id")["label"]

train_copy_pairs = [
    (TRAIN_PROCESSED_ROOT / f"{image_id}.jpg", LOCAL_PROCESSED_ROOT / f"{image_id}.jpg")
    for image_id in fold_assignment["image_id"]
]
train_copy_errors = parallel_copy(train_copy_pairs, desc="salin train ke lokal")
failed_train_ids = {int(Path(e["dest"]).stem) for e in train_copy_errors}
if train_copy_errors:
    print(f"PERINGATAN: {len(train_copy_errors)} gambar train gagal/timeout disalin -- dilewati.")

HARD_TEST_IDS = [6, 14, 19, 60, 116, 132, 293, 312, 363, 372, 413, 438, 499, 508, 565, 637, 659,
                  728, 781, 797, 843, 899, 907, 908, 1033, 1035, 1044, 1079, 1092, 1105, 1136,
                  1141, 1144, 1168, 1199, 1351, 1373, 1440]
test_copy_pairs = [
    (TEST_ROOT / f"{test_id}.jpg", LOCAL_TEST_ROOT / f"{test_id}.jpg") for test_id in HARD_TEST_IDS
]
parallel_copy(test_copy_pairs, desc="salin test ke lokal")
print(f"Selesai staging: {len(fold_assignment) - len(failed_train_ids)} gambar train + {len(HARD_TEST_IDS)} gambar test.")


## Model embedding: backbone ImageNet-pretrained (num_classes=0)

`num_classes=0` di timm mengganti classifier dengan Identity -- `forward()` langsung mengembalikan
fitur ter-pool, bukan logits 3 kelas. Dipilih checkpoint yang SAMA seperti yang dipakai ConvNeXt di
02a, tapi TANPA memuat bobot hasil fine-tune -- sengaja tetap murni ImageNet-pretrained, supaya ruang
fiturnya tidak "dipaksa" cuma memisahkan 3 kelas kompetisi (yang berisiko menghapus perbedaan
kertas-vs-bunga-asli yang justru ingin kita cari di sini).


In [ ]:
EMBED_MODEL_TAG = "convnextv2_base.fcmae_ft_in22k_in1k_384"

embed_model = timm.create_model(EMBED_MODEL_TAG, pretrained=True, num_classes=0).to(DEVICE).eval()
data_cfg = timm.data.resolve_model_data_config(embed_model)
mean, std, img_size = data_cfg["mean"], data_cfg["std"], data_cfg["input_size"][-1]
print(f"embedding dim: {embed_model.num_features}, img_size: {img_size}")

embed_tf = T.Compose([
    T.Resize(round(img_size / 0.95), interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(img_size),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std),
])


class ImageIdDataset(Dataset):
    def __init__(self, id_to_path: dict, transform):
        self.ids = list(id_to_path.keys())
        self.paths = list(id_to_path.values())
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            im = im.convert("RGB")
            tensor = self.transform(im)
        return tensor, self.ids[idx]


## Ekstrak embedding untuk seluruh gambar train (sekali saja)


In [ ]:
train_id_to_path = {
    int(image_id): LOCAL_PROCESSED_ROOT / f"{image_id}.jpg"
    for image_id in fold_assignment["image_id"] if image_id not in failed_train_ids
}
train_ds = ImageIdDataset(train_id_to_path, embed_tf)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

train_embeddings = np.zeros((len(train_ds), embed_model.num_features), dtype=np.float32)
train_ids_ordered = np.zeros(len(train_ds), dtype=np.int64)

ptr = 0
with torch.no_grad():
    for images, ids in tqdm(train_loader, desc="embedding train"):
        feats = F.normalize(embed_model(images.to(DEVICE)), dim=1).cpu().numpy()
        n = len(ids)
        train_embeddings[ptr:ptr + n] = feats
        train_ids_ordered[ptr:ptr + n] = ids.numpy()
        ptr += n

print(f"Selesai: {train_embeddings.shape[0]} embedding train tersimpan (dim={train_embeddings.shape[1]}).")


## Untuk tiap gambar test bermasalah: cari K tetangga train terdekat (cosine similarity)


In [ ]:
@torch.no_grad()
def embed_one(path):
    with Image.open(path) as im:
        im = im.convert("RGB")
        x = embed_tf(im).unsqueeze(0).to(DEVICE)
    feat = F.normalize(embed_model(x), dim=1)
    return feat.cpu().numpy()[0]


K = 5  # jumlah tetangga train terdekat yang dicek per gambar test

neighbor_records = {}  # test_id -> list of (train_id, cosine_similarity, label)
summary_rows = []
for test_id in tqdm(HARD_TEST_IDS, desc="cari tetangga"):
    test_feat = embed_one(LOCAL_TEST_ROOT / f"{test_id}.jpg")
    sims = train_embeddings @ test_feat  # sudah dinormalisasi -> dot product = cosine similarity
    top_idx = np.argsort(-sims)[:K]
    neighbors = [(int(train_ids_ordered[i]), float(sims[i]), int(label_by_id.loc[int(train_ids_ordered[i])]))
                 for i in top_idx]
    neighbor_records[test_id] = neighbors

    labels_nearest = [n[2] for n in neighbors]
    label_counts = pd.Series(labels_nearest).map(label_names).value_counts().to_dict()
    summary_rows.append({
        "test_id": test_id,
        "max_cosine_sim": round(max(s for _, s, _ in neighbors), 3),
        "cosine_sims": [round(s, 3) for _, s, _ in neighbors],
        "label_tetangga": label_counts,
    })

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print(summary_df.to_string(index=False))


## Bukti visual — query vs tetangga terdekatnya

Ini bukti valid yang sebenarnya: nilai cosine similarity di atas cuma pendukung, penilaian akhir
"apakah tetangga ini benar-benar mirip secara konsep/material" harus dilihat langsung dengan mata.
Tiap baris: gambar test (kotak merah) di kolom pertama, lalu 5 tetangga train terdekat + labelnya.
Disimpan juga ke Drive (`Preprocessing_Output/diagnostik_hard_cases/`) supaya bisa dilihat ulang
tanpa re-run.


In [ ]:
import matplotlib.pyplot as plt


def show_evidence(test_id, neighbors, thumb_size=(160, 160), save=True):
    n_cols = 1 + len(neighbors)
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 2.3, 2.8))

    with Image.open(LOCAL_TEST_ROOT / f"{test_id}.jpg") as im:
        im = im.convert("RGB")
        im.thumbnail(thumb_size)
        axes[0].imshow(im)
    axes[0].set_title(f"TEST {test_id}\n(QUERY)", fontsize=9, color="crimson", fontweight="bold")
    axes[0].axis("off")

    for ax, (train_id, sim, label) in zip(axes[1:], neighbors):
        with Image.open(LOCAL_PROCESSED_ROOT / f"{train_id}.jpg") as im:
            im = im.convert("RGB")
            im.thumbnail(thumb_size)
            ax.imshow(im)
        ax.set_title(f"train {train_id}\n{label_names[label]} (sim={sim:.2f})", fontsize=8)
        ax.axis("off")

    fig.suptitle(f"Test id={test_id}: query vs {len(neighbors)} tetangga train terdekat", fontsize=10)
    fig.tight_layout()
    if save:
        fig.savefig(EVIDENCE_OUTPUT_ROOT / f"evidence_test_{test_id}.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for test_id in HARD_TEST_IDS:
    show_evidence(test_id, neighbor_records[test_id])


## Interpretasi

Nilai patokan kasar untuk cosine similarity fitur ConvNeXtV2 ImageNet-pretrained:
- **> 0.6-0.7 DAN gambarnya memang terlihat serupa secara konsep/material** (dinilai dari grid di
  atas) -> data train punya sinyal cukup -> pola ini seharusnya bisa dipelajari lebih baik lewat
  augmentasi/durasi training, bukan ketiadaan data.
- **Similarity rendah ATAU gambar "tetangga" ternyata tidak nyambung sama sekali secara visual
  walau similarity-nya tinggi** -> data train tidak punya cakupan yang cukup untuk pola spesifik
  ini -> batasan data, bukan batasan teknik training.
- **Tetangga konsep/material-nya mirip TAPI labelnya beda-beda** -> ada inkonsistensi label untuk
  pola ini di data train sendiri.

Isi kesimpulan akhir di sini setelah melihat grid gambar di atas satu per satu.
